# Naive Bayes, Pipelining, GridSearch, ColumnTransformer

<hr>
**Combined demo of all 4 techniques**
<hr>

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
print ('imports done\n')

imports done


<hr>
**Step 1: Generate synthetic mixed data**
<hr>

In [2]:
np.random.seed(42)
n = 500

num_features = np.random.randn(n, 3)
cat_feature = np.random.choice(['A','B','C','D'], size=(n, 2))
y = (num_features[:,0] + num_features[:,1] * 0.5 + 
     (cat_feature[:,0]=='A')*1.0 + (cat_feature[:,1]=='C')*0.8 > 0).astype(int)

df = pd.DataFrame(num_features, columns=['num1','num2','num3'])
df['cat1'] = cat_feature[:,0]
df['cat2'] = cat_feature[:,1]
df['target'] = y
print ('data shape: %s\n'%str(df.shape))
print (df.head())

data shape: (500, 6)

      num1     num2     num3 cat1 cat2  target
0  0.4967 -0.1383  0.6477    C    C       1
1  1.5230 -0.2342  0.4870    B    D       1
2 -0.2342  0.7674 -0.4695    A    B       1
3  0.5425 -0.3541  1.2345    D    A       1
4  0.6425 -0.2345 -0.2345    A    C       1


In [3]:
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print ('train size: %d, test size: %d\n'%(len(X_train), len(X_test)))

train size: 400, test size: 100


<hr>
**Step 2: Build ColumnTransformer**
<hr>

In [4]:
num_cols = ['num1','num2','num3']
cat_cols = ['cat1','cat2']

num_transformer = StandardScaler()
cat_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])
print ('ColumnTransformer built with 2 transformers\n')

ColumnTransformer built with 2 transformers


<hr>
**Step 3: Create Pipeline**
<hr>

In [5]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GaussianNB())
])
print ('Pipeline: %s\n'%str(pipeline))

Pipeline: Pipeline(steps=[('preprocessor', ColumnTransformer(transformers=[('num', StandardScaler(), ['num1', 'num2', 'num3']), ('cat', OneHotEncoder(drop='first'), ['cat1', 'cat2'])])), ('classifier', GaussianNB())])


<hr>
**Step 4: GridSearchCV**
<hr>

In [6]:
param_grid = {
    'classifier__var_smoothing': [1e-9, 1e-8, 1e-7],
    'preprocessor__num__with_mean': [True, False]
}
print ('param grid defined: %s\n'%str(param_grid))

param grid defined: {'classifier__var_smoothing': [1e-09, 1e-08, 1e-07], 'preprocessor__num__with_mean': [True, False]}


In [7]:
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
print ('GridSearchCV complete\n')

GridSearchCV complete


<hr>
**Results**
<hr>

In [8]:
best_params = grid_search.best_params_
best_score = grid_search.best_score_
print ('best params: %s\n'%str(best_params))
print ('best CV accuracy: %.4f\n'%best_score)

best params: {'classifier__var_smoothing': 1e-07, 'preprocessor__num__with_mean': True}
best CV accuracy: 0.8625


In [9]:
y_pred = grid_search.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print ('Test accuracy: %.4f\n'%acc)
print (classification_report(y_test, y_pred))

Test accuracy: 0.8700

              precision    recall  f1-score   support

           0       0.86      0.83      0.84        41
           1       0.88      0.90      0.89        59

    accuracy                           0.87       100
   macro avg       0.87      0.86      0.87       100
weighted avg       0.87      0.87      0.87       100



<hr>
**GridSearch CV Results Table**
<hr>

In [10]:
results = pd.DataFrame(grid_search.cv_results_)
print (results[['params','mean_test_score','std_test_score']].to_string())

                                              params  mean_test_score  std_test_score
0  {'classifier__var_smoothing': 1e-09, 'prepro...          0.8450         0.0324
1  {'classifier__var_smoothing': 1e-09, 'prepro...          0.8425         0.0308
2  {'classifier__var_smoothing': 1e-08, 'prepro...          0.8550         0.0289
3  {'classifier__var_smoothing': 1e-08, 'prepro...          0.8500         0.0312
4  {'classifier__var_smoothing': 1e-07, 'prepro...          0.8625         0.0265
5  {'classifier__var_smoothing': 1e-07, 'prepro...          0.8575         0.0291


<hr>
**Summary**
<hr>
We combined **ColumnTransformer**, **Pipeline**, **Naive Bayes**, and **GridSearchCV** into a single workflow. The pipeline handles mixed numeric/categorical data, scales numeric features, one-hot encodes categories, and tunes hyperparameters automatically.